# Quantum ML Lab — Comparative Analysis

Loads all experiments from `logs/experiments.jsonl` and compares results.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

LOGS = Path("../logs/experiments.jsonl")

records = []
if LOGS.exists():
    with open(LOGS) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

print(f"{len(records)} experiments loaded")

In [ ]:
if records:
    df = pd.DataFrame([{
        "exp_id": r["exp_id"],
        "model": r["model_name"],
        "accuracy": r.get("final_acc"),
        "elapsed_s": r.get("elapsed_s"),
        "n_epochs": r.get("n_epochs"),
    } for r in records])
    display(df.sort_values("accuracy", ascending=False))

    fig = px.bar(df, x="model", y="accuracy", color="exp_id",
                 title="Accuracy by Model and Experiment")
    fig.show()

    fig2 = px.scatter(df, x="elapsed_s", y="accuracy", color="exp_id",
                      text="model", title="Efficiency: accuracy vs training time")
    fig2.show()
else:
    print("No experiments yet. Run exp_001 first.")

In [ ]:
# Learning curves — one trace per model
fig3 = go.Figure()
for r in records:
    if r.get("history") and r["history"] and "epoch" in r["history"][0]:
        epochs = [h["epoch"] for h in r["history"]]
        accs   = [h.get("accuracy", 0) for h in r["history"]]
        fig3.add_trace(go.Scatter(x=epochs, y=accs, name=r["model_name"], mode="lines"))

fig3.update_layout(title="Learning Curves", xaxis_title="Epoch", yaxis_title="Accuracy")
fig3.show()